# TerraMind Multitemporal Rice Prediction

Notebook นี้เป็น **Production-style full-image prediction workflow** สำหรับตรวจจับพื้นที่ข้าวจาก Sentinel-2 หลายช่วงเวลา โดยใช้ TerraMind checkpoint ที่ผ่าน fine-tuning แล้ว
---

## เหตุผลที่ Prediction ต้องใช้ fine-tuned checkpoint

TerraMind pretrained จาก Hugging Face รู้ representation ของข้อมูล Earth Observation แต่ยังไม่ได้เรียนรู้ว่า pixel ใดคือ “ข้าว” สำหรับ dataset ของ 

ดังนั้น:

```text
Hugging Face pretrained = จุดเริ่มต้นก่อน fine-tune
Fine-tuned checkpoint    = model ที่ใช้จำแนก rice จริง
```

Notebook นี้พยายามสร้าง architecture โดยไม่ดาวน์โหลด pretrained ซ้ำ แล้วโหลด rice checkpoint ด้วย `strict=True`  
หาก TerraTorch version หรือ checkpoint ต้องใช้ pretrained cache ระบบจึง fallback ไป Hugging Face

---

## Input contract

โครงสร้าง default:

```text
PROJECT_ROOT/
└── AOI_output/
    └── processed_2026/
        ├── stacks_10m/
        │   ├── ...20260313...STACK....tif
        │   ├── ...20260326...STACK....tif
        │   └── ...
        └── valid_masks_10m/
            ├── ...20260313...VALID_MASK....tif
            └── ...
```

Showcase stack มีอย่างน้อย 10 bands และโมเดลใช้:

```text
B02, B03, B04, B8A, B11, B12
```

Default fallback indexes:

```text
1, 2, 3, 8, 9, 10
```

# 0 — ก่อนเริ่ม

เลือก GPU:

```text
Runtime → Change runtime type → GPU
```

สำหรับ full image `10980 × 10980`, patch `256`, stride `128`:

```text
85 × 85 = 7,225 patches
```

จำนวนนี้เกิดจาก overlap 50% และตั้งใจใช้เพื่อให้รอยต่อระหว่าง patch เนียนขึ้น

GPU ที่แนะนำ:

- T4: ใช้ได้
- L4: เร็วกว่า T4
- A100: เร็วและใช้ batch ใหญ่ขึ้นได้

Notebook จะไม่ fallback ไป CPU เพื่อป้องกันงานที่ใช้เวลาหลายชั่วโมงโดยไม่ตั้งใจ

In [ ]:
# ============================================================
# 01 — INSTALL DEPENDENCIES
# ============================================================

%pip -q install --upgrade --upgrade-strategy only-if-needed --no-cache-dir \
    "terratorch==1.2.10" \
    "lightning>=2.5,<2.7" \
    "huggingface_hub>=0.27" \
    "rasterio>=1.3.9" \
    "geopandas>=0.14" \
    "shapely>=2.0" \
    "pyproj>=3.6" \
    "fiona>=1.9" \
    "scipy>=1.11" \
    "pandas>=2.1" \
    "matplotlib>=3.8" \
    "tqdm>=4.66"

# 1 — Import และตรวจ GPU

Cell นี้แก้ปัญหาที่เคยเกิดขึ้น:

- `terratorch.__version__` ไม่มี → ใช้ `importlib.metadata.version`
- PyTorch เป็น `+cpu` → หยุดทันที
- `DEVICE` หรือ `GPU_MEMORY_GB` ไม่ถูกประกาศ → สร้างใน cell เดียว
- mixed precision เลือกตาม GPU capability

In [ ]:
# ============================================================
# 02 — IMPORT + GPU ENVIRONMENT
# ============================================================

from __future__ import annotations

from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import copy
import gc
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import time
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
import terratorch

from huggingface_hub import hf_hub_download
from rasterio.features import shapes, sieve
from rasterio.windows import Window
from shapely.geometry import shape
from tqdm.auto import tqdm


def package_version(package_name: str) -> str:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "unknown"


print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("TerraTorch:", package_version("terratorch"))
print("Rasterio:", rasterio.__version__)
print("GeoPandas:", gpd.__version__)


if not torch.cuda.is_available():
    raise RuntimeError(
        "ไม่พบ CUDA GPU\n"
        "เลือก Runtime > Change runtime type > GPU แล้วเริ่มใหม่"
    )


DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
)
GPU_CAPABILITY = torch.cuda.get_device_capability(0)

AMP_DTYPE = (
    torch.bfloat16
    if GPU_CAPABILITY[0] >= 8
    else torch.float16
)

torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print("GPU:", GPU_NAME)
print("GPU memory:", f"{GPU_MEMORY_GB:.2f} GB")
print("AMP dtype:", AMP_DTYPE)

# 2 — Mount Google Drive

ผู้ใช้งานต้อง Add shortcut ของ project folder เข้า My Drive ก่อน

เมื่อ mount แล้ว path จะอยู่ใต้:

```text
/content/drive/MyDrive/
```

In [ ]:
# ============================================================
# 03 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=True,
)

print("Google Drive mounted")

# 3 — User configuration

แก้เฉพาะ cell นี้เมื่อเปลี่ยน project, model หรือ input

## Default showcase

- ใช้ project folder ที่แชร์
- model/config จาก `terramind_rice_multitemporal_final`
- input จาก `processed_2026`
- full image
- overlap 50%
- calibrated threshold จาก training config
- cache input ไป local SSD
- resume enabled

## เปลี่ยน dataset ในอนาคต

แก้:

```python
PROJECT_ROOT
MODEL_DIR
PROCESSED_ROOT
STACK_DIR
VALID_MASK_DIR
```

และตรวจให้ทุก timestamp มี grid เดียวกัน

In [ ]:
# ============================================================
# 04 — USER CONFIGURATION
# ============================================================

# ---------- Shared project ----------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Rice_Detect_TerraMind"
)

# ---------- Fine-tuned model ----------
MODEL_DIR = (
    PROJECT_ROOT
    / "Rice_TerraMind_Training"
    / "terramind_rice_multitemporal_final"
)

# ตั้ง None เพื่อเลือกอัตโนมัติ
MANUAL_CHECKPOINT_PATH = None

# ---------- Prediction input ----------
PROCESSED_ROOT = (
    PROJECT_ROOT
    / "AOI_output"
    / "processed_2026"
)

STACK_DIR = PROCESSED_ROOT / "stacks_10m"
VALID_MASK_DIR = PROCESSED_ROOT / "valid_masks_10m"

STACK_GLOB = "*STACK*.tif"
VALID_MASK_GLOB = "*.tif"

# ---------- Output ----------
RUN_NAME = "rice_fullimage_best_overlap50_final"

DRIVE_RUN_DIR = (
    PROJECT_ROOT
    / "Rice_Prediction"
    / RUN_NAME
)

# ---------- Quality ----------
PATCH_SIZE = 256
STRIDE = 128              # 50% overlap
USE_CONFIG_THRESHOLD = True
MANUAL_RICE_THRESHOLD = 0.50
MIN_PATCH_VALID_RATIO = 0.05

# ---------- Input behavior ----------
TIMESTAMP_POLICY = "training_exact"
# "training_exact" = ใช้วันที่เดียวกับ training config
# "all_available"  = ใช้ทุก stack ที่พบ เรียงตามวันที่

CACHE_INPUTS_TO_LOCAL = True
LOCAL_ROOT = Path("/content/rice_prediction")
LOCAL_INPUT_DIR = LOCAL_ROOT / "input"
LOCAL_MODEL_DIR = LOCAL_ROOT / "model"

# ---------- Resume ----------
RESUME_IF_AVAILABLE = True
RESET_INCOMPATIBLE_RESUME = True
SAVE_RESUME_EVERY_Y_ROW = 1

# ---------- Model loading ----------
PREFER_OFFLINE_CHECKPOINT_LOAD = True
ALLOW_HF_FALLBACK = True

HF_REPO_ID = "ibm-esa-geospatial/TerraMind-1.0-base"
HF_FILENAME = "TerraMind_v1_base.pt"
HF_CACHE_ROOT = PROJECT_ROOT / "_huggingface_cache"
HF_DOWNLOAD_RETRIES = 5

# ---------- Post-processing ----------
DO_SIEVE = True
MIN_OBJECT_AREA_M2 = 3000

# Vectorization อาจช้าและสร้าง polygon จำนวนมาก
EXPORT_VECTOR = False
SIMPLIFY_TOLERANCE_M = 5.0

# ---------- Delivery ----------
CREATE_PREVIEW = True
CREATE_ZIP = True
PLAY_AUDIO_WHEN_DONE = True


print("Project root:", PROJECT_ROOT)
print("Model directory:", MODEL_DIR)
print("Stack directory:", STACK_DIR)
print("Output:", DRIVE_RUN_DIR)
print("Patch:", PATCH_SIZE)
print("Stride:", STRIDE)
print(
    "Overlap:",
    f"{100 * (PATCH_SIZE - STRIDE) / PATCH_SIZE:.1f}%",
)

# 4 — ตรวจ path และเลือก checkpoint

ลำดับเลือก checkpoint:

1. `MANUAL_CHECKPOINT_PATH`
2. `best.ckpt`
3. `best-rice-iou-XX.ckpt` ที่ epoch สูงสุด
4. `last.ckpt`

Prediction ใช้ best checkpoint เป็นหลัก ส่วน `last.ckpt` เหมาะกับ resume training

ไฟล์ที่ต้องมี:

```text
checkpoint
training_config.json
normalization_stats.json
stacks_10m/
```

In [ ]:
# ============================================================
# 05 — PATHS + CHECKPOINT SELECTION
# ============================================================

for required_path in [
    PROJECT_ROOT,
    MODEL_DIR,
    STACK_DIR,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"ไม่พบ path:\n{required_path}\n"
            "ตรวจ PROJECT_ROOT และ shortcut ใน My Drive"
        )


def extract_epoch(path: Path) -> int:
    patterns = [
        r"epoch[=_-]?(\d+)",
        r"rice-iou-(\d+)",
        r"(\d+)(?=\.ckpt$)",
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            path.name,
            flags=re.IGNORECASE,
        )

        if match:
            return int(match.group(1))

    return -1


def select_prediction_checkpoint() -> Path:
    if MANUAL_CHECKPOINT_PATH is not None:
        path = Path(MANUAL_CHECKPOINT_PATH)

        if not path.exists():
            raise FileNotFoundError(path)

        return path

    canonical_best = MODEL_DIR / "best.ckpt"

    if canonical_best.exists():
        return canonical_best

    historical_best = sorted(
        MODEL_DIR.glob("*rice*iou*.ckpt"),
        key=extract_epoch,
    )

    if historical_best:
        return historical_best[-1]

    last_path = MODEL_DIR / "last.ckpt"

    if last_path.exists():
        return last_path

    raise FileNotFoundError(
        f"ไม่พบ checkpoint ใน {MODEL_DIR}"
    )


CHECKPOINT_DRIVE = select_prediction_checkpoint()
TRAINING_CONFIG_DRIVE = MODEL_DIR / "training_config.json"
NORMALIZATION_DRIVE = MODEL_DIR / "normalization_stats.json"

for required_path in [
    CHECKPOINT_DRIVE,
    TRAINING_CONFIG_DRIVE,
    NORMALIZATION_DRIVE,
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)


with open(
    TRAINING_CONFIG_DRIVE,
    "r",
    encoding="utf-8",
) as file:
    training_config = json.load(file)

with open(
    NORMALIZATION_DRIVE,
    "r",
    encoding="utf-8",
) as file:
    normalization_stats = json.load(file)


BAND_NAMES = list(
    normalization_stats["band_names"]
)

BAND_MEANS = np.asarray(
    normalization_stats["means"],
    dtype=np.float32,
)

BAND_STDS = np.asarray(
    normalization_stats["stds"],
    dtype=np.float32,
)

TRAINING_TIMESTAMPS = [
    str(value)
    for value in training_config.get(
        "timestamps",
        [],
    )
]

RICE_THRESHOLD = (
    float(
        training_config.get(
            "recommended_probability_threshold",
            MANUAL_RICE_THRESHOLD,
        )
    )
    if USE_CONFIG_THRESHOLD
    else float(MANUAL_RICE_THRESHOLD)
)


if BAND_MEANS.shape != (len(BAND_NAMES),):
    raise RuntimeError("Normalization means shape mismatch")

if BAND_STDS.shape != (len(BAND_NAMES),):
    raise RuntimeError("Normalization stds shape mismatch")


print("Checkpoint:", CHECKPOINT_DRIVE)
print("Checkpoint epoch:", extract_epoch(CHECKPOINT_DRIVE))
print("Training timestamps:", TRAINING_TIMESTAMPS)
print("Bands:", BAND_NAMES)
print("Rice threshold:", RICE_THRESHOLD)

# 5 — ค้นหา multitemporal stacks

Notebook ค้นหาเฉพาะ `STACK_DIR` จึงไม่หยิบไฟล์ `indices_10m` ที่มี 5 bands มาผิดอีก

วันที่อ่านจากชื่อไฟล์ด้วย pattern:

```text
20YYYYMMDD
```

`training_exact` บังคับให้วันที่ตรงกับ Fine-tune Notebook  
`all_available` เหมาะเมื่อ architecture รองรับ variable T และผู้ใช้ยอมรับความต่างด้าน phenology

In [ ]:
# ============================================================
# 06 — FIND STACKS + VALID MASKS
# ============================================================

DATE_PATTERN = re.compile(r"(20\d{6})")


def extract_date(path: Path) -> str | None:
    match = DATE_PATTERN.search(path.name)
    return match.group(1) if match else None


stack_by_date = {}

for path in sorted(STACK_DIR.glob(STACK_GLOB)):
    date = extract_date(path)

    if date:
        stack_by_date[date] = path


if not stack_by_date:
    raise FileNotFoundError(
        f"ไม่พบ stack ด้วย pattern {STACK_GLOB} ใน {STACK_DIR}"
    )


mask_by_date = {}

if VALID_MASK_DIR.exists():
    for path in sorted(
        VALID_MASK_DIR.glob(VALID_MASK_GLOB)
    ):
        date = extract_date(path)

        if date:
            mask_by_date[date] = path


if TIMESTAMP_POLICY == "training_exact":
    if not TRAINING_TIMESTAMPS:
        raise RuntimeError(
            "training_config ไม่มี timestamps"
        )

    TIMESTAMPS = list(TRAINING_TIMESTAMPS)

elif TIMESTAMP_POLICY == "all_available":
    TIMESTAMPS = sorted(stack_by_date)

else:
    raise ValueError(
        "TIMESTAMP_POLICY ต้องเป็น training_exact หรือ all_available"
    )


missing_dates = [
    date
    for date in TIMESTAMPS
    if date not in stack_by_date
]

if missing_dates:
    raise FileNotFoundError(
        f"ขาด stacks สำหรับวันที่: {missing_dates}"
    )


IMAGE_PATHS_DRIVE = [
    stack_by_date[date]
    for date in TIMESTAMPS
]

MASK_PATHS_DRIVE = [
    mask_by_date.get(date)
    for date in TIMESTAMPS
]


print("Selected timestamps:", len(TIMESTAMPS))

for date, image_path, mask_path in zip(
    TIMESTAMPS,
    IMAGE_PATHS_DRIVE,
    MASK_PATHS_DRIVE,
):
    print(
        date,
        "| image:",
        image_path.name,
        "| mask:",
        (
            mask_path.name
            if mask_path is not None
            else "AUTO"
        ),
    )

# 6 — Copy input และ checkpoint ไป local SSD

Full-image inference อ่านหน้าต่างหลายพันครั้ง  
การอ่าน random windows จาก Google Drive mount มี latency สูงกว่าการอ่านจาก local SSD

Cell นี้:

- รวมขนาดไฟล์ที่ต้องใช้
- ตรวจพื้นที่ว่าง
- copy เฉพาะ stacks, masks, checkpoint และ configs ที่เลือก
- ตรวจ byte size หลัง copy
- fallback ไป Drive เมื่อพื้นที่ไม่พอ

Output และ resume state ยังคงเขียนกลับ Google Drive

In [ ]:
# ============================================================
# 07 — LOCAL INPUT CACHE
# ============================================================

LOCAL_INPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)


def copy_file_verified(
    source: Path,
    destination: Path,
) -> Path:
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if (
        destination.exists()
        and destination.stat().st_size
        == source.stat().st_size
    ):
        print("Cached:", destination.name)
        return destination

    temporary = destination.with_suffix(
        destination.suffix + ".partial"
    )

    if temporary.exists():
        temporary.unlink()

    try:
        subprocess.run(
            [
                "rsync",
                "-ah",
                "--info=progress2",
                str(source),
                str(temporary),
            ],
            check=True,
        )
    except Exception:
        shutil.copy2(source, temporary)

    if temporary.stat().st_size != source.stat().st_size:
        raise IOError(
            f"Copy size mismatch: {source}"
        )

    temporary.replace(destination)

    print("Copied:", destination.name)
    return destination


selected_files = (
    IMAGE_PATHS_DRIVE
    + [
        path
        for path in MASK_PATHS_DRIVE
        if path is not None
    ]
    + [
        CHECKPOINT_DRIVE,
        TRAINING_CONFIG_DRIVE,
        NORMALIZATION_DRIVE,
    ]
)

required_bytes = sum(
    path.stat().st_size
    for path in selected_files
)

free_bytes = shutil.disk_usage("/content").free
safety_margin_bytes = 8 * 1024**3

print(
    "Selected input size:",
    f"{required_bytes / 1024**3:.2f} GB",
)

print(
    "Local free space:",
    f"{free_bytes / 1024**3:.2f} GB",
)


USE_LOCAL_INPUTS = (
    CACHE_INPUTS_TO_LOCAL
    and free_bytes > required_bytes + safety_margin_bytes
)


if USE_LOCAL_INPUTS:
    IMAGE_PATHS = [
        copy_file_verified(
            source,
            LOCAL_INPUT_DIR / source.name,
        )
        for source in IMAGE_PATHS_DRIVE
    ]

    MASK_PATHS = [
        (
            copy_file_verified(
                source,
                LOCAL_INPUT_DIR / source.name,
            )
            if source is not None
            else None
        )
        for source in MASK_PATHS_DRIVE
    ]

    CHECKPOINT_PATH = copy_file_verified(
        CHECKPOINT_DRIVE,
        LOCAL_MODEL_DIR / CHECKPOINT_DRIVE.name,
    )

else:
    IMAGE_PATHS = IMAGE_PATHS_DRIVE
    MASK_PATHS = MASK_PATHS_DRIVE
    CHECKPOINT_PATH = CHECKPOINT_DRIVE


print("Using local input cache:", USE_LOCAL_INPUTS)
print("Active checkpoint:", CHECKPOINT_PATH)

# 7 — เปิดไฟล์และตรวจ grid แบบเร็ว

ตรวจเฉพาะ metadata ไม่ plot ภาพเต็ม:

- band count
- width / height
- CRS
- transform
- resolution
- valid mask grid

ไม่มี Session preview input ที่อ่านภาพเต็มและทำให้เสียเวลา

In [ ]:
# ============================================================
# 08 — OPEN + VALIDATE GRID
# ============================================================

IMAGE_SOURCES = [
    rasterio.open(path)
    for path in IMAGE_PATHS
]

MASK_SOURCES = [
    (
        rasterio.open(path)
        if path is not None
        else None
    )
    for path in MASK_PATHS
]


reference = IMAGE_SOURCES[0]

FULL_WIDTH = reference.width
FULL_HEIGHT = reference.height
FULL_TRANSFORM = reference.transform
FULL_CRS = reference.crs
FULL_RESOLUTION = reference.res
FULL_PROFILE = reference.profile.copy()


for source, path in zip(
    IMAGE_SOURCES,
    IMAGE_PATHS,
):
    if source.width != FULL_WIDTH:
        raise RuntimeError(
            f"Width mismatch: {path.name}"
        )

    if source.height != FULL_HEIGHT:
        raise RuntimeError(
            f"Height mismatch: {path.name}"
        )

    if source.transform != FULL_TRANSFORM:
        raise RuntimeError(
            f"Transform mismatch: {path.name}"
        )

    if source.crs != FULL_CRS:
        raise RuntimeError(
            f"CRS mismatch: {path.name}"
        )


for source, date in zip(
    MASK_SOURCES,
    TIMESTAMPS,
):
    if source is None:
        continue

    if (
        source.width != FULL_WIDTH
        or source.height != FULL_HEIGHT
        or source.transform != FULL_TRANSFORM
        or source.crs != FULL_CRS
    ):
        raise RuntimeError(
            f"Valid-mask grid mismatch: {date}"
        )


print("Full image:", FULL_WIDTH, "x", FULL_HEIGHT)
print("CRS:", FULL_CRS)
print("Resolution:", FULL_RESOLUTION)
print("Timestamps:", len(IMAGE_SOURCES))
print("Grid validation: PASSED")

# 8 — Resolve band indexes

ลำดับการหา bands:

1. ใช้ band descriptions ใน GeoTIFF ถ้ามีและ match ได้ครบ
2. หาก stack มีอย่างน้อย 10 bands ใช้ mapping showcase:
   `B02=1, B03=2, B04=3, B8A=8, B11=9, B12=10`
3. หากไม่เข้าเงื่อนไขให้หยุด ไม่เดาไฟล์ 5-band indices

เมื่อต้องใช้ dataset ใหม่ ให้ใส่ band descriptions ที่ชัดเจน หรือแก้ fallback mapping ใน cell นี้

In [ ]:
# ============================================================
# 09 — BAND MAPPING
# ============================================================

DEFAULT_SOURCE_BAND_INDEX = {
    "B02": 1,
    "B03": 2,
    "B04": 3,
    "B8A": 8,
    "B11": 9,
    "B12": 10,
}

NAME_ALIASES = {
    "B02": {"B02", "BLUE"},
    "B03": {"B03", "GREEN"},
    "B04": {"B04", "RED"},
    "B8A": {"B8A", "NIR_NARROW"},
    "B11": {"B11", "SWIR_1", "SWIR1"},
    "B12": {"B12", "SWIR_2", "SWIR2"},
}


def normalize_description(value) -> str:
    if value is None:
        return ""

    return re.sub(
        r"[^A-Z0-9_]",
        "",
        str(value).upper(),
    )


def resolve_band_indexes(source) -> list[int]:
    descriptions = [
        normalize_description(value)
        for value in source.descriptions
    ]

    resolved = []

    for band_name in BAND_NAMES:
        aliases = {
            normalize_description(alias)
            for alias in NAME_ALIASES.get(
                band_name,
                {band_name},
            )
        }

        found_index = None

        for index, description in enumerate(
            descriptions,
            start=1,
        ):
            if description in aliases:
                found_index = index
                break

        if found_index is None:
            resolved = []
            break

        resolved.append(found_index)

    if resolved:
        return resolved

    if (
        all(
            band_name in DEFAULT_SOURCE_BAND_INDEX
            for band_name in BAND_NAMES
        )
        and source.count
        >= max(
            DEFAULT_SOURCE_BAND_INDEX[
                band_name
            ]
            for band_name in BAND_NAMES
        )
    ):
        return [
            DEFAULT_SOURCE_BAND_INDEX[
                band_name
            ]
            for band_name in BAND_NAMES
        ]

    raise ValueError(
        "ไม่สามารถ map bands ได้\n"
        f"Model bands: {BAND_NAMES}\n"
        f"Source band count: {source.count}\n"
        f"Descriptions: {source.descriptions}"
    )


BAND_INDEXES = resolve_band_indexes(reference)

MODEL_BAND_TO_SOURCE_INDEX = dict(
    zip(BAND_NAMES, BAND_INDEXES)
)

for source, path in zip(
    IMAGE_SOURCES,
    IMAGE_PATHS,
):
    if source.count < max(BAND_INDEXES):
        raise RuntimeError(
            f"{path.name} มีเพียง {source.count} bands"
        )


print("Model bands:", BAND_NAMES)
print("Source indexes:", BAND_INDEXES)
print("Band mapping: PASSED")

# 9 — Full-image patch grid

`PATCH_SIZE=256` คือ model เห็นพื้นที่ครั้งละ 256×256 pixels

`STRIDE=128` คือเลื่อนครั้งละ 128 pixels จึง overlap 50%

สำหรับภาพ 10980×10980:

```text
85 positions แนวนอน
85 positions แนวตั้ง
7,225 patches
```

Cell นี้สร้าง output profile จาก source image โดยตรง และ assert ว่า output ต้องมีขนาดเต็ม  
ไม่มีตัวแปร crop จึงไม่สามารถเกิดปัญหา predict 7,225 patches แต่เขียน output 2048×2048 แบบเดิม

In [ ]:
# ============================================================
# 10 — FULL-IMAGE GRID + OUTPUT PROFILE
# ============================================================

def make_steps(
    length: int,
    patch_size: int,
    stride: int,
) -> list[int]:
    if stride > patch_size:
        raise ValueError(
            "STRIDE ห้ามมากกว่า PATCH_SIZE เพราะจะเกิดช่องว่าง"
        )

    if length <= patch_size:
        return [0]

    steps = list(
        range(
            0,
            length - patch_size + 1,
            stride,
        )
    )

    last = length - patch_size

    if steps[-1] != last:
        steps.append(last)

    return steps


X_STEPS = make_steps(
    FULL_WIDTH,
    PATCH_SIZE,
    STRIDE,
)

Y_STEPS = make_steps(
    FULL_HEIGHT,
    PATCH_SIZE,
    STRIDE,
)

TOTAL_PATCHES = (
    len(X_STEPS)
    * len(Y_STEPS)
)


FULL_OUTPUT_PROFILE = FULL_PROFILE.copy()

FULL_OUTPUT_PROFILE.update(
    width=FULL_WIDTH,
    height=FULL_HEIGHT,
    transform=FULL_TRANSFORM,
    crs=FULL_CRS,
)


assert FULL_OUTPUT_PROFILE["width"] == FULL_WIDTH
assert FULL_OUTPUT_PROFILE["height"] == FULL_HEIGHT
assert FULL_OUTPUT_PROFILE["transform"] == FULL_TRANSFORM
assert STRIDE <= PATCH_SIZE


if FULL_WIDTH == 10980 and FULL_HEIGHT == 10980:
    if PATCH_SIZE == 256 and STRIDE == 128:
        assert TOTAL_PATCHES == 7225


print("FULL IMAGE MODE")
print("Input:", FULL_WIDTH, "x", FULL_HEIGHT)
print("Output:", FULL_OUTPUT_PROFILE["width"], "x", FULL_OUTPUT_PROFILE["height"])
print("X positions:", len(X_STEPS))
print("Y positions:", len(Y_STEPS))
print("Total patches:", TOTAL_PATCHES)
print(
    "Overlap:",
    f"{100 * (PATCH_SIZE - STRIDE) / PATCH_SIZE:.1f}%",
)
print("Full-image configuration: PASSED")

# 10 — Load fine-tuned model

ลำดับ:

1. อ่าน architecture จาก Lightning checkpoint
2. ปิดเฉพาะ `backbone_pretrained` key ที่มีอยู่จริง
3. สร้าง architecture โดยไม่ติดต่อ Hugging Face
4. โหลด fine-tuned weights ด้วย `strict=True`
5. หาก offline reconstruction ไม่สำเร็จ จึง cache pretrained จาก Hugging Face และ retry

Notebook **ไม่ส่ง `pretrained=False`** เข้า EncoderDecoderFactory เพราะ TerraTorch แจ้งว่า argument นี้ไม่ได้ถูกใช้

In [ ]:
# ============================================================
# 11 — MODEL LOADING WITH SAFE HF FALLBACK
# ============================================================

os.environ["HF_HOME"] = str(HF_CACHE_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_ROOT / "hub")

HF_TOKEN = None

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")


def ensure_terramind_cached() -> Path:
    last_error = None

    for attempt in range(1, HF_DOWNLOAD_RETRIES + 1):
        try:
            path = hf_hub_download(
                repo_id=HF_REPO_ID,
                filename=HF_FILENAME,
                cache_dir=os.environ["HF_HUB_CACHE"],
                token=HF_TOKEN,
                force_download=False,
            )

            print("HF cache ready:", path)
            return Path(path)

        except Exception as error:
            last_error = error
            wait_seconds = min(2 ** attempt, 30)

            print(
                f"HF attempt {attempt}/{HF_DOWNLOAD_RETRIES} failed:",
                type(error).__name__,
            )

            if attempt < HF_DOWNLOAD_RETRIES:
                time.sleep(wait_seconds)

    raise RuntimeError(
        "ไม่สามารถเตรียม TerraMind pretrained cache"
    ) from last_error


def disable_existing_pretrained_flags(value):
    if isinstance(value, dict):
        for key in list(value.keys()):
            if key in {
                "backbone_pretrained",
                "pretrained_backbone",
            }:
                value[key] = False
            else:
                disable_existing_pretrained_flags(
                    value[key]
                )

    elif isinstance(value, list):
        for item in value:
            disable_existing_pretrained_flags(item)


def load_finetuned_model(
    checkpoint_path: Path,
):
    offline_error = None

    if PREFER_OFFLINE_CHECKPOINT_LOAD:
        try:
            checkpoint = torch.load(
                checkpoint_path,
                map_location="cpu",
                weights_only=False,
            )

            hparams = checkpoint.get(
                "hyper_parameters",
                {},
            )

            if "model_args" not in hparams:
                raise RuntimeError(
                    "Checkpoint ไม่มี hyper_parameters['model_args']"
                )

            model_args = copy.deepcopy(
                hparams["model_args"]
            )

            disable_existing_pretrained_flags(
                model_args
            )

            load_kwargs = {
                "model_args": model_args,
            }

            if "model_factory" in hparams:
                load_kwargs["model_factory"] = (
                    hparams["model_factory"]
                )

            del checkpoint
            gc.collect()

            loaded_model = (
                terratorch.tasks.SemanticSegmentationTask
                .load_from_checkpoint(
                    str(checkpoint_path),
                    map_location="cpu",
                    strict=True,
                    **load_kwargs,
                )
            )

            print("Model loaded without HF download")
            return loaded_model

        except Exception as error:
            offline_error = error

            print(
                "Offline checkpoint reconstruction failed:",
                type(error).__name__,
                str(error)[:300],
            )

    if not ALLOW_HF_FALLBACK:
        raise RuntimeError(
            "Offline model load failed and HF fallback is disabled"
        ) from offline_error

    ensure_terramind_cached()

    loaded_model = (
        terratorch.tasks.SemanticSegmentationTask
        .load_from_checkpoint(
            str(checkpoint_path),
            map_location="cpu",
            strict=True,
        )
    )

    print("Model loaded with HF cache fallback")
    return loaded_model


model = load_finetuned_model(
    CHECKPOINT_PATH
)

model = model.to(DEVICE)
model.eval()

print("Model device:", DEVICE)
print("Checkpoint strict load: PASSED")

# 11 — Adaptive GPU batch

Batch เริ่มต้นเลือกตาม VRAM:

- T4 15 GB → 2
- L4/A10 20–24 GB → 4
- A100 40 GB → 8
- A100 80 GB → 16

หากเกิด CUDA out-of-memory ฟังก์ชันจะแบ่ง batch ครึ่งหนึ่ง recursively แทนการทำให้งานทั้งหมดพัง

In [ ]:
# ============================================================
# 12 — GPU BATCH PREDICTION
# ============================================================

if GPU_MEMORY_GB >= 70:
    INITIAL_BATCH_SIZE = 16
elif GPU_MEMORY_GB >= 38:
    INITIAL_BATCH_SIZE = 8
elif GPU_MEMORY_GB >= 20:
    INITIAL_BATCH_SIZE = 4
else:
    INITIAL_BATCH_SIZE = 2


def autocast_context():
    return torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
        enabled=True,
    )


def predict_batch_recursive(
    batch_array: np.ndarray,
) -> np.ndarray:
    try:
        batch = torch.from_numpy(
            batch_array
        ).pin_memory().to(
            DEVICE,
            non_blocking=True,
        )

        with torch.inference_mode():
            with autocast_context():
                result = model(batch)

                logits = (
                    result.output
                    if hasattr(result, "output")
                    else result
                )

                probabilities = torch.softmax(
                    logits,
                    dim=1,
                )[:, 1]

        output = (
            probabilities
            .float()
            .cpu()
            .numpy()
        )

        del (
            batch,
            result,
            logits,
            probabilities,
        )

        return output

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gc.collect()

        if len(batch_array) <= 1:
            raise RuntimeError(
                "GPU memory ไม่พอแม้ batch size = 1"
            )

        middle = len(batch_array) // 2

        return np.concatenate(
            [
                predict_batch_recursive(
                    batch_array[:middle]
                ),
                predict_batch_recursive(
                    batch_array[middle:]
                ),
            ],
            axis=0,
        )


print("Initial GPU batch size:", INITIAL_BATCH_SIZE)
print(
    "predict_batch_recursive ready:",
    callable(predict_batch_recursive),
)

# 12 — Patch reading, valid masks และ temporal imputation

แต่ละ patch อ่าน:

```text
T timestamps × 6 bands × 256 × 256
```

หาก valid mask มี จะใช้ mask นั้น  
หากไม่มี จะสร้าง valid mask จาก pixel ที่ไม่เป็น 0 และไม่เป็น 65535

Temporal median imputation:

- pixel ที่ invalid ในบางวัน → ใช้ median จากวันที่ valid
- invalid ทุกวัน → ใช้ 0
- warning `All-NaN slice encountered` ถูก suppress เฉพาะจุดนี้

In [ ]:
# ============================================================
# 13 — PATCH READING + PREPROCESSING
# ============================================================

def make_blend_weight(size: int) -> np.ndarray:
    one_dimensional = np.hanning(
        size
    ).astype(np.float32)

    weight = np.outer(
        one_dimensional,
        one_dimensional,
    )

    return np.maximum(
        weight,
        0.05,
    ).astype(np.float32)


BLEND_WEIGHT = make_blend_weight(
    PATCH_SIZE
)


def temporal_median_impute(
    images: np.ndarray,
    valid_masks: np.ndarray,
) -> np.ndarray:
    images_float = images.astype(
        np.float32,
        copy=False,
    )

    valid_bool = valid_masks.astype(
        bool,
        copy=False,
    )

    masked = np.where(
        valid_bool[:, None, :, :],
        images_float,
        np.nan,
    )

    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore",
            category=RuntimeWarning,
        )

        median = np.nanmedian(
            masked,
            axis=0,
        )

    median = np.nan_to_num(
        median,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    output = images_float.copy()

    for time_index in range(images.shape[0]):
        invalid = ~valid_bool[time_index]

        output[time_index] = np.where(
            invalid[None, :, :],
            median,
            output[time_index],
        )

    return output


def read_patch(
    x: int,
    y: int,
):
    window = Window(
        x,
        y,
        PATCH_SIZE,
        PATCH_SIZE,
    )

    images = []
    masks = []

    for image_source, mask_source in zip(
        IMAGE_SOURCES,
        MASK_SOURCES,
    ):
        # ไม่ใช้ out_shape เพื่อหลีกเลี่ยง NumPy 2.5 deprecation warning
        image = image_source.read(
            indexes=BAND_INDEXES,
            window=window,
        ).astype(
            np.float32,
            copy=False,
        )

        if image.shape != (
            len(BAND_INDEXES),
            PATCH_SIZE,
            PATCH_SIZE,
        ):
            raise RuntimeError(
                f"Unexpected patch shape: {image.shape}"
            )

        if mask_source is not None:
            mask_data = mask_source.read(
                1,
                window=window,
            )

            valid = mask_data > 0
        else:
            valid = np.all(
                image > 0,
                axis=0,
            )

            valid &= np.all(
                image != 65535,
                axis=0,
            )

        images.append(image)
        masks.append(
            valid.astype(
                np.uint8,
                copy=False,
            )
        )

    return (
        np.stack(images, axis=0),
        np.stack(masks, axis=0),
    )


def preprocess_patch(
    raw_images: np.ndarray,
    valid_masks: np.ndarray,
) -> np.ndarray:
    images = temporal_median_impute(
        raw_images,
        valid_masks,
    )

    # [T,C,H,W] -> [C,T,H,W]
    images = np.transpose(
        images,
        (1, 0, 2, 3),
    )

    images = (
        images
        - BAND_MEANS[:, None, None, None]
    ) / BAND_STDS[:, None, None, None]

    return np.ascontiguousarray(
        images,
        dtype=np.float32,
    )

# 13 — Preflight smoke test

ก่อนเริ่ม 7,225 patches ต้องผ่านทุกข้อ:

- ฟังก์ชันครบ
- patch อ่านได้
- input shape ถูก
- model output shape ถูก
- probability finite
- output profile เท่าภาพเต็ม

หากผิด Notebook จะหยุดก่อนเสียเวลา

In [ ]:
# ============================================================
# 14 — PREFLIGHT SMOKE TEST
# ============================================================

required_objects = {
    "model": model,
    "DEVICE": DEVICE,
    "AMP_DTYPE": AMP_DTYPE,
    "INITIAL_BATCH_SIZE": INITIAL_BATCH_SIZE,
    "predict_batch_recursive": predict_batch_recursive,
    "read_patch": read_patch,
    "preprocess_patch": preprocess_patch,
}

for name, value in required_objects.items():
    if value is None:
        raise RuntimeError(f"{name} is missing")


test_raw_images, test_valid_masks = read_patch(
    X_STEPS[0],
    Y_STEPS[0],
)

test_image = preprocess_patch(
    test_raw_images,
    test_valid_masks,
)

test_batch = test_image[None, ...]
test_probability = predict_batch_recursive(
    test_batch
)


expected_input_shape = (
    1,
    len(BAND_NAMES),
    len(TIMESTAMPS),
    PATCH_SIZE,
    PATCH_SIZE,
)

if test_batch.shape != expected_input_shape:
    raise RuntimeError(
        "Input shape mismatch\n"
        f"Expected: {expected_input_shape}\n"
        f"Found: {test_batch.shape}"
    )

if test_probability.shape != (
    1,
    PATCH_SIZE,
    PATCH_SIZE,
):
    raise RuntimeError(
        f"Output probability shape mismatch: {test_probability.shape}"
    )

if not np.isfinite(test_probability).all():
    raise RuntimeError(
        "Probability contains NaN or Inf"
    )

assert FULL_OUTPUT_PROFILE["width"] == FULL_WIDTH
assert FULL_OUTPUT_PROFILE["height"] == FULL_HEIGHT

print("Input shape:", test_batch.shape)
print("Probability shape:", test_probability.shape)
print(
    "Probability range:",
    float(test_probability.min()),
    float(test_probability.max()),
)
print("Full output:", FULL_WIDTH, "x", FULL_HEIGHT)
print("Total patches:", TOTAL_PATCHES)
print("Preflight: PASSED")

del (
    test_raw_images,
    test_valid_masks,
    test_image,
    test_batch,
    test_probability,
)

gc.collect()
torch.cuda.empty_cache()

# 14 — Resume system และ run signature

Resume state ผูกกับ:

- checkpoint
- width/height
- transform
- timestamps
- bands
- patch size
- stride
- threshold

ถ้า configuration เปลี่ยน จะไม่ใช้ resume state เก่าโดยไม่ตั้งใจ

ไฟล์ระหว่างรัน:

```text
rice_probability.partial.tif
rice_raw_mask.partial.tif
resume_buffers.npz
resume_state.json
```

หลังสำเร็จจะเปลี่ยนเป็น final outputs และลบ resume state

In [ ]:
# ============================================================
# 15 — RUN SIGNATURE + OUTPUT PATHS
# ============================================================

signature_payload = json.dumps(
    {
        "checkpoint": CHECKPOINT_DRIVE.name,
        "checkpoint_size": CHECKPOINT_DRIVE.stat().st_size,
        "width": FULL_WIDTH,
        "height": FULL_HEIGHT,
        "transform": tuple(FULL_TRANSFORM),
        "crs": str(FULL_CRS),
        "timestamps": TIMESTAMPS,
        "bands": BAND_NAMES,
        "band_indexes": BAND_INDEXES,
        "patch_size": PATCH_SIZE,
        "stride": STRIDE,
        "threshold": RICE_THRESHOLD,
    },
    sort_keys=True,
).encode("utf-8")

RUN_SIGNATURE = hashlib.sha256(
    signature_payload
).hexdigest()[:20]


PARTIAL_PROBABILITY_PATH = (
    DRIVE_RUN_DIR
    / "rice_probability.partial.tif"
)

PARTIAL_RAW_MASK_PATH = (
    DRIVE_RUN_DIR
    / "rice_raw_mask.partial.tif"
)

RESUME_BUFFER_PATH = (
    DRIVE_RUN_DIR
    / "resume_buffers.npz"
)

RESUME_META_PATH = (
    DRIVE_RUN_DIR
    / "resume_state.json"
)


PROBABILITY_OUTPUT = (
    DRIVE_RUN_DIR
    / "rice_probability_full.tif"
)

RAW_MASK_OUTPUT = (
    DRIVE_RUN_DIR
    / "rice_raw_mask_full.tif"
)

CLEAN_MASK_OUTPUT = (
    DRIVE_RUN_DIR
    / "rice_clean_mask_full.tif"
)

VECTOR_OUTPUT = (
    DRIVE_RUN_DIR
    / "rice_parcels_full.gpkg"
)

SUMMARY_OUTPUT = (
    DRIVE_RUN_DIR
    / "prediction_summary_full.csv"
)

MANIFEST_OUTPUT = (
    DRIVE_RUN_DIR
    / "run_manifest.json"
)

PREVIEW_OUTPUT = (
    DRIVE_RUN_DIR
    / "prediction_preview_full.png"
)


print("Run signature:", RUN_SIGNATURE)
print("Output directory:", DRIVE_RUN_DIR)

# 15 — Resumable streaming full-image inference

หลักการ blending:

- patch ซ้อนกัน 50%
- Hann weight ให้ความสำคัญกลาง patch มากกว่าขอบ
- probability จาก patch ที่ซ้อนกันถูก weighted average

หลักการ streaming:

1. ประมวลผล patch row หนึ่งแถว
2. finalize raster rows ที่จะไม่มี patch ในอนาคตมาซ้อน
3. เขียนลง partial GeoTIFF
4. shift rolling buffer
5. บันทึก resume state

ดังนั้น runtime หลุดหลังจบแต่ละ Y-row สามารถเริ่มต่อได้

In [ ]:
# ============================================================
# 16 — RESUMABLE FULL-IMAGE INFERENCE
# ============================================================

PROBABILITY_SCALE = 10000
PROBABILITY_NODATA = 65535


probability_profile = (
    FULL_OUTPUT_PROFILE.copy()
)

probability_profile.update(
    driver="GTiff",
    count=1,
    dtype="uint16",
    nodata=PROBABILITY_NODATA,
    compress="DEFLATE",
    predictor=2,
    tiled=True,
    blockxsize=512,
    blockysize=512,
    BIGTIFF="YES",
)


mask_profile = (
    FULL_OUTPUT_PROFILE.copy()
)

mask_profile.update(
    driver="GTiff",
    count=1,
    dtype="uint8",
    nodata=255,
    compress="DEFLATE",
    predictor=2,
    tiled=True,
    blockxsize=512,
    blockysize=512,
    BIGTIFF="YES",
)


resume_files = [
    PARTIAL_PROBABILITY_PATH,
    PARTIAL_RAW_MASK_PATH,
    RESUME_BUFFER_PATH,
    RESUME_META_PATH,
]


def remove_resume_files():
    for path in resume_files:
        if path.exists():
            path.unlink()


resume_is_valid = (
    RESUME_IF_AVAILABLE
    and all(path.exists() for path in resume_files)
)


if resume_is_valid:
    with open(
        RESUME_META_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        resume_meta = json.load(file)

    if resume_meta.get("run_signature") != RUN_SIGNATURE:
        if RESET_INCOMPATIBLE_RESUME:
            print(
                "Existing resume state is incompatible; starting a new run"
            )
            remove_resume_files()
            resume_is_valid = False
        else:
            raise RuntimeError(
                "Resume state belongs to another configuration"
            )


if resume_is_valid:
    with np.load(RESUME_BUFFER_PATH) as arrays:
        rolling_sum = arrays[
            "rolling_sum"
        ].astype(np.float32, copy=True)

        rolling_weight = arrays[
            "rolling_weight"
        ].astype(np.float32, copy=True)

    START_Y_INDEX = int(
        resume_meta["next_y_index"]
    )

    processed_patches = int(
        resume_meta.get("processed_patches", 0)
    )

    skipped_patches = int(
        resume_meta.get("skipped_patches", 0)
    )

    print(
        "Resuming from Y row:",
        START_Y_INDEX,
        "/",
        len(Y_STEPS),
    )

else:
    rolling_sum = np.zeros(
        (PATCH_SIZE, FULL_WIDTH),
        dtype=np.float32,
    )

    rolling_weight = np.zeros(
        (PATCH_SIZE, FULL_WIDTH),
        dtype=np.float32,
    )

    START_Y_INDEX = 0
    processed_patches = 0
    skipped_patches = 0

    remove_resume_files()

    with rasterio.open(
        PARTIAL_PROBABILITY_PATH,
        "w",
        **probability_profile,
    ):
        pass

    with rasterio.open(
        PARTIAL_RAW_MASK_PATH,
        "w",
        **mask_profile,
    ):
        pass


def atomic_save_resume(
    next_y_index: int,
):
    temporary_buffer = (
        RESUME_BUFFER_PATH.parent
        / "resume_buffers.npz.partial"
    )

    temporary_meta = (
        RESUME_META_PATH.parent
        / "resume_state.json.partial"
    )

    with open(temporary_buffer, "wb") as file:
        np.savez(
            file,
            rolling_sum=rolling_sum,
            rolling_weight=rolling_weight,
        )

    state = {
        "run_signature": RUN_SIGNATURE,
        "next_y_index": int(next_y_index),
        "processed_patches": int(processed_patches),
        "skipped_patches": int(skipped_patches),
        "total_patches": int(TOTAL_PATCHES),
        "updated_at": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
    }

    with open(
        temporary_meta,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            state,
            file,
            indent=2,
        )

    temporary_buffer.replace(
        RESUME_BUFFER_PATH
    )

    temporary_meta.replace(
        RESUME_META_PATH
    )


def flush_pending_batch(
    pending_images: list[np.ndarray],
    pending_x_positions: list[int],
):
    if not pending_images:
        return 0, None, None

    batch_array = np.stack(
        pending_images,
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )

    probabilities = predict_batch_recursive(
        batch_array
    )

    for probability, x in zip(
        probabilities,
        pending_x_positions,
    ):
        rolling_sum[
            :PATCH_SIZE,
            x:x + PATCH_SIZE,
        ] += probability * BLEND_WEIGHT

        rolling_weight[
            :PATCH_SIZE,
            x:x + PATCH_SIZE,
        ] += BLEND_WEIGHT

    count = len(pending_images)
    input_shape = batch_array.shape
    output_shape = probabilities.shape

    pending_images.clear()
    pending_x_positions.clear()

    del batch_array, probabilities

    return count, input_shape, output_shape


remaining_patches = (
    (len(Y_STEPS) - START_Y_INDEX)
    * len(X_STEPS)
)

progress = tqdm(
    total=remaining_patches,
    desc="Full-image rice prediction",
)

inference_start_time = time.time()
first_batch_printed = False


for y_index in range(
    START_Y_INDEX,
    len(Y_STEPS),
):
    y = Y_STEPS[y_index]
    row_start_time = time.time()

    pending_images = []
    pending_x_positions = []

    for x in X_STEPS:
        raw_images, valid_masks = read_patch(
            x,
            y,
        )

        usable_ratio = float(
            np.any(
                valid_masks == 1,
                axis=0,
            ).mean()
        )

        if usable_ratio < MIN_PATCH_VALID_RATIO:
            skipped_patches += 1
            progress.update(1)
            continue

        image = preprocess_patch(
            raw_images,
            valid_masks,
        )

        pending_images.append(image)
        pending_x_positions.append(x)

        if len(pending_images) >= INITIAL_BATCH_SIZE:
            count, input_shape, output_shape = flush_pending_batch(
                pending_images,
                pending_x_positions,
            )

            processed_patches += count

            if not first_batch_printed:
                print("First GPU batch input:", input_shape)
                print("First GPU batch output:", output_shape)
                first_batch_printed = True

        progress.update(1)

    count, input_shape, output_shape = flush_pending_batch(
        pending_images,
        pending_x_positions,
    )

    processed_patches += count

    if not first_batch_printed and input_shape is not None:
        print("First GPU batch input:", input_shape)
        print("First GPU batch output:", output_shape)
        first_batch_printed = True


    if y_index + 1 < len(Y_STEPS):
        rows_to_finalize = (
            Y_STEPS[y_index + 1] - y
        )
    else:
        rows_to_finalize = FULL_HEIGHT - y


    numerator = rolling_sum[
        :rows_to_finalize
    ]

    denominator = rolling_weight[
        :rows_to_finalize
    ]

    valid = denominator > 0

    probability = np.zeros(
        numerator.shape,
        dtype=np.float32,
    )

    probability[valid] = (
        numerator[valid]
        / denominator[valid]
    )

    stored_probability = np.full(
        probability.shape,
        PROBABILITY_NODATA,
        dtype=np.uint16,
    )

    stored_probability[valid] = np.clip(
        np.rint(
            probability[valid]
            * PROBABILITY_SCALE
        ),
        0,
        PROBABILITY_SCALE,
    ).astype(np.uint16)

    raw_mask = np.full(
        probability.shape,
        255,
        dtype=np.uint8,
    )

    raw_mask[valid] = (
        probability[valid]
        >= RICE_THRESHOLD
    ).astype(np.uint8)


    output_window = Window(
        0,
        y,
        FULL_WIDTH,
        rows_to_finalize,
    )

    with rasterio.open(
        PARTIAL_PROBABILITY_PATH,
        "r+",
    ) as destination:
        destination.write(
            stored_probability,
            1,
            window=output_window,
        )

    with rasterio.open(
        PARTIAL_RAW_MASK_PATH,
        "r+",
    ) as destination:
        destination.write(
            raw_mask,
            1,
            window=output_window,
        )


    remaining_rows = (
        PATCH_SIZE - rows_to_finalize
    )

    if remaining_rows > 0:
        rolling_sum[:remaining_rows] = rolling_sum[
            rows_to_finalize:
            rows_to_finalize + remaining_rows
        ].copy()

        rolling_weight[:remaining_rows] = rolling_weight[
            rows_to_finalize:
            rows_to_finalize + remaining_rows
        ].copy()

    rolling_sum[remaining_rows:] = 0
    rolling_weight[remaining_rows:] = 0


    if (
        (y_index + 1)
        % SAVE_RESUME_EVERY_Y_ROW
        == 0
        and y_index + 1 < len(Y_STEPS)
    ):
        atomic_save_resume(
            y_index + 1
        )


    row_elapsed = time.time() - row_start_time
    completed_rows = y_index - START_Y_INDEX + 1
    remaining_rows_count = len(Y_STEPS) - y_index - 1
    estimated_remaining_seconds = (
        row_elapsed * remaining_rows_count
    )

    print(
        f"Completed Y row {y_index + 1}/{len(Y_STEPS)}"
        f" | row time {row_elapsed / 60:.2f} min"
        f" | estimated remaining "
        f"{estimated_remaining_seconds / 60:.1f} min"
    )


progress.close()

TOTAL_INFERENCE_SECONDS = (
    time.time() - inference_start_time
)

print("Processed patches:", processed_patches)
print("Skipped patches:", skipped_patches)
print(
    "Inference time:",
    f"{TOTAL_INFERENCE_SECONDS / 60:.2f} minutes",
)
print("Full-image inference complete: True")

# 16 — Finalize probability และ raw mask

เมื่อ inference เสร็จ:

- partial probability → `rice_probability_full.tif`
- partial raw mask → `rice_raw_mask_full.tif`
- assert dimensions, CRS และ transform
- resume files ยังเก็บไว้จน post-processing สำเร็จ

In [ ]:
# ============================================================
# 17 — FINALIZE PROBABILITY + RAW MASK
# ============================================================

for final_path in [
    PROBABILITY_OUTPUT,
    RAW_MASK_OUTPUT,
]:
    if final_path.exists():
        final_path.unlink()


shutil.copy2(
    PARTIAL_PROBABILITY_PATH,
    PROBABILITY_OUTPUT,
)

shutil.copy2(
    PARTIAL_RAW_MASK_PATH,
    RAW_MASK_OUTPUT,
)


for output_path in [
    PROBABILITY_OUTPUT,
    RAW_MASK_OUTPUT,
]:
    with rasterio.open(output_path) as source:
        if source.width != FULL_WIDTH:
            raise RuntimeError(
                f"Output width mismatch: {output_path.name}"
            )

        if source.height != FULL_HEIGHT:
            raise RuntimeError(
                f"Output height mismatch: {output_path.name}"
            )

        if source.transform != FULL_TRANSFORM:
            raise RuntimeError(
                f"Output transform mismatch: {output_path.name}"
            )

        if source.crs != FULL_CRS:
            raise RuntimeError(
                f"Output CRS mismatch: {output_path.name}"
            )


print("Probability:", PROBABILITY_OUTPUT)
print("Raw mask:", RAW_MASK_OUTPUT)
print("Full-size output verification: PASSED")

# 17 — Post-processing

Raw mask มาจาก probability threshold โดยตรง

Clean mask ใช้ `sieve` เพื่อลบกลุ่ม rice ที่เล็กกว่า:

```python
MIN_OBJECT_AREA_M2 = 3000
```

Sentinel-2 10 m มี pixel area ประมาณ 100 m²  
ดังนั้น 3000 m² เท่ากับประมาณ 30 pixels

Nodata `255` ถูกเก็บไว้

In [ ]:
# ============================================================
# 18 — CLEAN MASK
# ============================================================

with rasterio.open(
    RAW_MASK_OUTPUT
) as source:
    raw_mask = source.read(1)


valid_output = raw_mask != 255
rice_binary = raw_mask == 1

pixel_area_m2 = abs(
    FULL_RESOLUTION[0]
    * FULL_RESOLUTION[1]
)

minimum_pixels = max(
    1,
    int(
        math.ceil(
            MIN_OBJECT_AREA_M2
            / pixel_area_m2
        )
    ),
)


if DO_SIEVE:
    clean_rice = sieve(
        rice_binary.astype(np.uint8),
        size=minimum_pixels,
        connectivity=8,
    )
else:
    clean_rice = rice_binary.astype(np.uint8)


clean_mask = np.full(
    raw_mask.shape,
    255,
    dtype=np.uint8,
)

clean_mask[valid_output] = clean_rice[
    valid_output
]


with rasterio.open(
    CLEAN_MASK_OUTPUT,
    "w",
    **mask_profile,
) as destination:
    destination.write(clean_mask, 1)

    destination.set_band_description(
        1,
        "RICE_CLEAN_MASK_FULL",
    )


with rasterio.open(
    CLEAN_MASK_OUTPUT
) as source:
    assert source.width == FULL_WIDTH
    assert source.height == FULL_HEIGHT
    assert source.transform == FULL_TRANSFORM
    assert source.crs == FULL_CRS


print("Minimum object pixels:", minimum_pixels)
print("Clean mask:", CLEAN_MASK_OUTPUT)
print("Clean-mask verification: PASSED")

# 18 — Optional vector export

Vectorization ปิดเป็น default เพราะ full image อาจสร้าง polygon จำนวนมาก

เปิดด้วย:

```python
EXPORT_VECTOR = True
```

Output เป็น GeoPackage และเก็บ:

- geometry
- area_m2
- area_rai

1 ไร่ = 1,600 m²

In [ ]:
# ============================================================
# 19 — OPTIONAL VECTOR EXPORT
# ============================================================

if EXPORT_VECTOR:
    geometries = []

    for geometry_mapping, value in shapes(
        clean_mask,
        mask=clean_mask == 1,
        transform=FULL_TRANSFORM,
        connectivity=8,
    ):
        if int(value) != 1:
            continue

        geometry = shape(
            geometry_mapping
        )

        if SIMPLIFY_TOLERANCE_M > 0:
            geometry = geometry.simplify(
                SIMPLIFY_TOLERANCE_M,
                preserve_topology=True,
            )

        if not geometry.is_empty:
            geometries.append(geometry)

    if geometries:
        rice_gdf = gpd.GeoDataFrame(
            {"class": ["rice"] * len(geometries)},
            geometry=geometries,
            crs=FULL_CRS,
        )

        rice_gdf["area_m2"] = rice_gdf.geometry.area
        rice_gdf["area_rai"] = (
            rice_gdf["area_m2"] / 1600.0
        )

        rice_gdf.to_file(
            VECTOR_OUTPUT,
            driver="GPKG",
        )

        print("Vector polygons:", len(rice_gdf))
        print("Vector output:", VECTOR_OUTPUT)

        del rice_gdf
    else:
        print("No rice polygon found")

    del geometries
    gc.collect()

else:
    print("Vector export disabled")

# 19 — Summary และ area statistics

คำนวณแบบ block เพื่อไม่โหลด output ซ้ำหลายครั้ง

Area:

```text
rice_area_m2
rice_area_km2
rice_area_rai
```

Summary เก็บ checkpoint, dates, bands, GPU, patch, stride, threshold และ runtime เพื่อ audit ย้อนหลัง

In [ ]:
# ============================================================
# 20 — SUMMARY + MANIFEST
# ============================================================

rice_pixels = int(
    np.count_nonzero(
        clean_mask == 1
    )
)

valid_pixels = int(
    np.count_nonzero(
        clean_mask != 255
    )
)

rice_area_m2 = (
    rice_pixels * pixel_area_m2
)

summary = {
    "project": "TerraMind Rice Detection",
    "run_name": RUN_NAME,
    "run_signature": RUN_SIGNATURE,
    "checkpoint": CHECKPOINT_DRIVE.name,
    "checkpoint_epoch": extract_epoch(
        CHECKPOINT_DRIVE
    ),
    "gpu": GPU_NAME,
    "gpu_memory_gb": GPU_MEMORY_GB,
    "terratorch_version": package_version("terratorch"),
    "timestamps": len(TIMESTAMPS),
    "timestamp_list": ",".join(TIMESTAMPS),
    "bands": ",".join(BAND_NAMES),
    "band_indexes": ",".join(
        str(value)
        for value in BAND_INDEXES
    ),
    "patch_size": PATCH_SIZE,
    "stride": STRIDE,
    "overlap_percent": (
        100
        * (PATCH_SIZE - STRIDE)
        / PATCH_SIZE
    ),
    "total_patches": TOTAL_PATCHES,
    "processed_patches": processed_patches,
    "skipped_patches": skipped_patches,
    "threshold": RICE_THRESHOLD,
    "minimum_object_area_m2": MIN_OBJECT_AREA_M2,
    "width": FULL_WIDTH,
    "height": FULL_HEIGHT,
    "crs": str(FULL_CRS),
    "pixel_area_m2": pixel_area_m2,
    "valid_pixels": valid_pixels,
    "rice_pixels": rice_pixels,
    "rice_area_m2": rice_area_m2,
    "rice_area_km2": rice_area_m2 / 1_000_000,
    "rice_area_rai": rice_area_m2 / 1600,
    "local_input_cache": USE_LOCAL_INPUTS,
    "inference_minutes": TOTAL_INFERENCE_SECONDS / 60,
    "completed_at": time.strftime(
        "%Y-%m-%d %H:%M:%S"
    ),
}


pd.DataFrame(
    [summary]
).to_csv(
    SUMMARY_OUTPUT,
    index=False,
)


with open(
    MANIFEST_OUTPUT,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2,
    )


display(
    pd.DataFrame([summary]).T.rename(
        columns={0: "value"}
    )
)

# 20 — Preview แบบ downsample

Preview ไม่อ่านภาพ RGB เต็ม resolution เข้า RAM

แสดง:

- Sentinel-2 RGB จาก timestamp กลาง
- clean rice mask overlay
- probability
- clean mask

ไฟล์ preview ไม่ใช่ข้อมูลหลัก ผลจริงคือ GeoTIFF

In [ ]:
# ============================================================
# 21 — OPTIONAL DOWNSAMPLED PREVIEW
# ============================================================

def stretch_rgb(rgb: np.ndarray) -> np.ndarray:
    output = np.zeros_like(
        rgb,
        dtype=np.float32,
    )

    for channel in range(3):
        low, high = np.nanpercentile(
            rgb[..., channel],
            [2, 98],
        )

        output[..., channel] = np.clip(
            (rgb[..., channel] - low)
            / max(high - low, 1e-6),
            0,
            1,
        )

    return output


if CREATE_PREVIEW:
    preview_max_size = 1400

    scale = min(
        preview_max_size / FULL_WIDTH,
        preview_max_size / FULL_HEIGHT,
        1.0,
    )

    preview_width = max(
        1,
        int(round(FULL_WIDTH * scale)),
    )

    preview_height = max(
        1,
        int(round(FULL_HEIGHT * scale)),
    )

    middle_source = IMAGE_SOURCES[
        len(IMAGE_SOURCES) // 2
    ]

    # Model band order ของ showcase คือ B02, B03, B04, B8A, B11, B12
    required_rgb_names = ["B04", "B03", "B02"]

    if not all(
        band_name in MODEL_BAND_TO_SOURCE_INDEX
        for band_name in required_rgb_names
    ):
        raise RuntimeError(
            "ไม่พบ B02/B03/B04 สำหรับสร้าง RGB preview"
        )

    rgb_source_indexes = [
        MODEL_BAND_TO_SOURCE_INDEX[band_name]
        for band_name in required_rgb_names
    ]

    rgb = middle_source.read(
        indexes=rgb_source_indexes,
        out_shape=(
            3,
            preview_height,
            preview_width,
        ),
        resampling=rasterio.enums.Resampling.bilinear,
    ).astype(np.float32)

    rgb = np.transpose(rgb, (1, 2, 0))
    rgb = stretch_rgb(rgb)

    with rasterio.open(
        PROBABILITY_OUTPUT
    ) as source:
        probability_preview = source.read(
            1,
            out_shape=(
                preview_height,
                preview_width,
            ),
            resampling=rasterio.enums.Resampling.bilinear,
        ).astype(np.float32)

    probability_preview[
        probability_preview == PROBABILITY_NODATA
    ] = np.nan

    probability_preview = (
        probability_preview
        / PROBABILITY_SCALE
    )

    with rasterio.open(
        CLEAN_MASK_OUTPUT
    ) as source:
        mask_preview = source.read(
            1,
            out_shape=(
                preview_height,
                preview_width,
            ),
            resampling=rasterio.enums.Resampling.nearest,
        )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(21, 7),
    )

    axes[0].imshow(rgb)
    axes[0].imshow(
        np.ma.masked_where(
            mask_preview != 1,
            mask_preview,
        ),
        alpha=0.55,
        vmin=0,
        vmax=1,
    )
    axes[0].set_title("RGB + Rice mask")
    axes[0].axis("off")

    axes[1].imshow(
        probability_preview,
        vmin=0,
        vmax=1,
    )
    axes[1].set_title("Rice probability")
    axes[1].axis("off")

    mask_show = mask_preview.astype(np.float32)
    mask_show[mask_preview == 255] = np.nan

    axes[2].imshow(
        mask_show,
        vmin=0,
        vmax=1,
    )
    axes[2].set_title("Clean rice mask")
    axes[2].axis("off")

    plt.tight_layout()

    fig.savefig(
        PREVIEW_OUTPUT,
        dpi=180,
        bbox_inches="tight",
    )

    plt.show()

    print("Preview:", PREVIEW_OUTPUT)

else:
    print("Preview disabled")

# 21 — Final verification

ตรวจว่า:

- output เปิดได้
- output เต็มภาพ
- CRS/transform ตรง input
- dtype/nodata ถูก
- probability, raw mask, clean mask มีจริง
- summary และ manifest มีจริง

จากนั้นจึงลบ resume state

In [ ]:
# ============================================================
# 22 — FINAL OUTPUT VERIFICATION
# ============================================================

required_outputs = [
    PROBABILITY_OUTPUT,
    RAW_MASK_OUTPUT,
    CLEAN_MASK_OUTPUT,
    SUMMARY_OUTPUT,
    MANIFEST_OUTPUT,
]

if CREATE_PREVIEW:
    required_outputs.append(
        PREVIEW_OUTPUT
    )


verification_rows = []

for path in required_outputs:
    if not path.exists():
        raise FileNotFoundError(path)

    row = {
        "file": path.name,
        "size_mb": path.stat().st_size / 1024**2,
        "status": "OK",
    }

    if path.suffix.lower() == ".tif":
        with rasterio.open(path) as source:
            if source.width != FULL_WIDTH:
                raise RuntimeError(
                    f"{path.name}: width mismatch"
                )

            if source.height != FULL_HEIGHT:
                raise RuntimeError(
                    f"{path.name}: height mismatch"
                )

            if source.crs != FULL_CRS:
                raise RuntimeError(
                    f"{path.name}: CRS mismatch"
                )

            if source.transform != FULL_TRANSFORM:
                raise RuntimeError(
                    f"{path.name}: transform mismatch"
                )

            row.update({
                "width": source.width,
                "height": source.height,
                "dtype": source.dtypes[0],
                "nodata": source.nodata,
                "crs": str(source.crs),
            })

    verification_rows.append(row)


verification_table = pd.DataFrame(
    verification_rows
)

display(verification_table)

for path in [
    RESUME_BUFFER_PATH,
    RESUME_META_PATH,
]:
    if path.exists():
        path.unlink()

print("Final verification: PASSED")

# 22 — Optional ZIP

ZIP สร้างจากไฟล์ final เท่านั้น ไม่รวม partial/resume files

สร้างใน local `/content` ก่อนแล้วจึง copy ไป Drive เพื่อหลีกเลี่ยง archive ซ้อนตัวเอง

In [ ]:
# ============================================================
# 23 — OPTIONAL RESULT ZIP
# ============================================================

ZIP_OUTPUT = None

if CREATE_ZIP:
    delivery_dir = (
        LOCAL_ROOT / "delivery"
    )

    if delivery_dir.exists():
        shutil.rmtree(delivery_dir)

    delivery_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    delivery_files = [
        PROBABILITY_OUTPUT,
        RAW_MASK_OUTPUT,
        CLEAN_MASK_OUTPUT,
        SUMMARY_OUTPUT,
        MANIFEST_OUTPUT,
    ]

    if CREATE_PREVIEW:
        delivery_files.append(
            PREVIEW_OUTPUT
        )

    if EXPORT_VECTOR and VECTOR_OUTPUT.exists():
        delivery_files.append(
            VECTOR_OUTPUT
        )

    for source in delivery_files:
        shutil.copy2(
            source,
            delivery_dir / source.name,
        )

    archive_base = (
        LOCAL_ROOT / RUN_NAME
    )

    ZIP_OUTPUT = Path(
        shutil.make_archive(
            str(archive_base),
            "zip",
            root_dir=delivery_dir,
        )
    )

    DRIVE_ZIP_OUTPUT = (
        DRIVE_RUN_DIR / ZIP_OUTPUT.name
    )

    shutil.copy2(
        ZIP_OUTPUT,
        DRIVE_ZIP_OUTPUT,
    )

    print("ZIP:", DRIVE_ZIP_OUTPUT)

else:
    print("ZIP disabled")

# 23 — Close resources และ notification

ปิด raster sources และคืน GPU memory  
Output อยู่ใน Google Drive แล้ว จึงไม่หายเมื่อ runtime ปิดหลังจากนี้

In [ ]:
# ============================================================
# 24 — CLOSE + DONE
# ============================================================

for source in IMAGE_SOURCES:
    source.close()

for source in MASK_SOURCES:
    if source is not None:
        source.close()

del (
    model,
    raw_mask,
    rice_binary,
    clean_rice,
    clean_mask,
)

gc.collect()
torch.cuda.empty_cache()

print("=" * 80)
print("PREDICTION WORKFLOW COMPLETE")
print("=" * 80)
print("Output directory:", DRIVE_RUN_DIR)
print("Probability:", PROBABILITY_OUTPUT)
print("Raw mask:", RAW_MASK_OUTPUT)
print("Clean mask:", CLEAN_MASK_OUTPUT)
print("Summary:", SUMMARY_OUTPUT)

if ZIP_OUTPUT is not None:
    print("ZIP:", DRIVE_RUN_DIR / ZIP_OUTPUT.name)


if PLAY_AUDIO_WHEN_DONE:
    try:
        from IPython.display import Audio, display

        sample_rate = 16000
        duration_seconds = 1.0

        time_axis = np.linspace(
            0,
            duration_seconds,
            int(sample_rate * duration_seconds),
            endpoint=False,
        )

        tone = 0.15 * np.sin(
            2 * np.pi * 880 * time_axis
        )

        display(
            Audio(
                tone,
                rate=sample_rate,
                autoplay=True,
            )
        )

    except Exception:
        pass

# Troubleshooting guide

## `ValueError: arguments {'pretrained': False} were passed but not used`

ห้ามเพิ่ม:

```python
model_args["pretrained"] = False
```

Notebook นี้แก้เฉพาะ key ที่มีอยู่จริง เช่น `backbone_pretrained`

## Hugging Face HTTP 504

Offline checkpoint reconstruction จะถูกลองก่อน  
HF ใช้เป็น fallback และมี retry/cache

## `predict_batch_recursive is not defined`

Notebook นี้ประกาศฟังก์ชันก่อน Preflight และตรวจ `callable()` ก่อนเริ่ม

## `batch is not defined`

Model forward อยู่ภายใน `predict_batch_recursive` ไม่มี code fragment แยก

## Stack ถูกอ่านเป็น 5 bands

Notebook ค้นหาเฉพาะ `STACK_DIR` และ require band mapping ที่ถูกต้อง

## Output ได้เพียง 2048×2048

Notebook ไม่มี crop mode  
`FULL_OUTPUT_PROFILE` สร้างจาก source และ assert ก่อนเริ่ม/หลังจบ

## Runtime หลุด

รันใหม่จากบนลงล่างโดยใช้ `RUN_NAME` เดิม  
ระบบอ่าน `resume_state.json` และทำต่อจาก Y-row ล่าสุด

## Resume incompatible

เปลี่ยน checkpoint, stride, threshold หรือ input แล้ว signature จะไม่ตรง  
default จะลบ partial state เก่าและเริ่ม run ใหม่

## CUDA out of memory

Batch จะถูก split ครึ่งอัตโนมัติ  
หาก batch size 1 ยังไม่พอจึงหยุด

## `All-NaN slice encountered`

ถูกจัดการใน temporal median imputation  
pixel ที่ invalid ทุกวันกลายเป็น 0 และ pixel ที่ไม่มี output weight เป็น nodata

# References

- TerraTorch official repository: https://github.com/torchgeo/terratorch
- TerraMind model card: https://huggingface.co/ibm-esa-geospatial/TerraMind-1.0-base
- TerraMind paper: https://arxiv.org/abs/2504.11171
- TerraTorch paper: https://arxiv.org/abs/2503.20563
- Lightning checkpointing: https://lightning.ai/docs/pytorch/stable/common/checkpointing_basic.html

Prediction result ควรได้รับการตรวจด้วย independent ground truth หรือ visual QA ก่อนใช้ตัดสินใจเชิงปฏิบัติการ